# GROUP PROJECT

**IMPORTANT from classes**

* try remove shit data: create 3 folds and record on what photos model fails - they are the condidats to be removed
* try ebmedinng model and find outliers in that space for each cluster
* mb try base models from hugging face 

* use tensorboard
* add a heatmap on top of image at the end
* plot the closest images

## Importing libraries and setting up google drive

In [ ]:
#
import numpy as np
import pandas as pd

# Data visualisation
import matplotlib.pyplot as plt
from matplotlib import style
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# sklearn related
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight

#
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.regularizers import L1, L2
#
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from keras.layers import Input, Dense, Dropout
from keras.optimizers import Adam, SGD, Adagrad, Adadelta, RMSprop
from keras.utils import to_categorical

# cnn related
from keras.layers import Conv2D, Flatten, MaxPooling2D, GlobalAveragePooling2D, BatchNormalization

from PIL import Image



# sort later :3
import tensorflow as tf
import math
from collections import Counter
from PIL import Image as PilImage

# manipulating files
import cv2
from tqdm import tqdm
import os
import ast

# For image showing
from IPython.display import Image, display
import matplotlib.pyplot as plt
import os


In [ ]:
directory = 'rare_species/'

In [ ]:
# Place this at the very top of your script/notebook
tf.config.optimizer.set_jit(True)
# Alternatively, compile your model with the jit_compile flag
# model.compile(jit_compile=True, ...)

In [ ]:
# Enable Mixed Precision policy at the start
from tensorflow.keras.mixed_precision import set_global_policy
set_global_policy('mixed_float16')
# Make sure the final output layer is still float32 for stability
# final_layer = Dense(num_classes, activation='softmax', dtype='float32')(previous_layer)

In [ ]:
# # Set up google drive FOR COLAB
# from google.colab.patches import cv2_imshow
# from google.colab import drive
# drive.mount('/content/drive/')

# # The directory where all the files will go
# !unzip -o "/content/drive/MyDrive/DeepLearningProject/rare_species.zip" -d {directory} > /dev/null # remove all the prints (takes +-2min yo run)

In [ ]:
# import torch
# print("CUDA available:", torch.cuda.is_available())
# print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

In [ ]:
print("TensorFlow version:", tf.__version__)
print("GPUs detected:", tf.config.list_physical_devices('GPU'))

In [ ]:
# tf.debugging.set_log_device_placement(True)     # to see which device is being used

In [ ]:
metadata = pd.read_csv(f'{directory}metadata.csv')

## Custom functions (later to .py)

In [ ]:
def explore_image_files(file_paths, explore_values=False):
    # Define the variables of smalles and biggest images
    image_sizes = []
    color_channels = []
    formats = []
    min_vals = []
    max_vals = []
    ratios = []

    # Iteration loop for each folder to compare the image sizes

    for file_path in file_paths:
        with PilImage.open(file_path) as img:
            image_sizes.append(img.size)
            color_channels.append(img.mode)
            formats.append(img.format)
            ratios.append(img.size[0]/img.size[1])

            if explore_values:
                # Convert to numpy to check the actual data type, Takes alot of time
                img_array = np.array(img)

                # Value range
                min_vals.append(img_array.min())
                max_vals.append(img_array.max())

    if explore_values:
        return image_sizes, color_channels, formats, ratios, min_vals, max_vals
    else:
        return image_sizes, color_channels, formats, ratios

## Initial exploration

### Metadata exploration

#### Basic exploration

In [ ]:
metadata.info()

In [ ]:
metadata.head()

In [ ]:
metadata.describe(include='O')

In [ ]:
# Get phylum counts
phylum_counts = metadata['phylum'].value_counts()

# Calculate number of families per phylum
families_per_phylum = metadata.groupby('phylum')['family'].nunique()

# Create custom hover data with family counts
hover_data = []
for phylum in phylum_counts.index:
    image_count = phylum_counts[phylum]
    family_count = families_per_phylum[phylum]
    percentage = (image_count / len(metadata)) * 100
    hover_data.append([image_count, percentage, family_count])

# Convert to numpy array for easy indexing
hover_data = np.array(hover_data)

# Create figure
fig = go.Figure(go.Bar(
    x=phylum_counts.index,
    y=phylum_counts.values,
    marker=dict(
        color='#6366f1',
        line=dict(color='#4f46e5', width=0.5)
    ),
    text=phylum_counts.values,
    textposition='outside',
    textfont=dict(size=12),
    hovertemplate='<b>%{x}</b><br>' +
                  'Images: %{y}<br>' +
                  'Families: %{customdata[2]}<br>' +
                  'Percentage: %{customdata[1]:.2f}%<extra></extra>',
    customdata=hover_data
))

fig.update_layout(
    title={
        'text': '<b>Distribution of Species by Phylum</b>',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 18}
    },
    xaxis_title='Phylum',
    yaxis_title='Count',
    height=600,
    width=1000,
    plot_bgcolor='white',
    paper_bgcolor='white',
    xaxis=dict(
        tickfont=dict(size=11),
        showgrid=False
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor='#e5e7eb',
        gridwidth=1
    ),
    font=dict(size=11),
    showlegend=False
)

fig.show()

# Print summary
print("\n" + "="*60)
print("PHYLUM DISTRIBUTION WITH FAMILY COUNTS")
print("="*60)
for phylum in phylum_counts.index:
    image_count = phylum_counts[phylum]
    family_count = families_per_phylum[phylum]
    pct = (image_count / len(metadata)) * 100
    avg_images_per_family = image_count / family_count
    print(f"{phylum:20s}: {image_count:4d} images ({pct:5.2f}%) | {family_count:3d} families | Avg: {avg_images_per_family:.1f} images/family")
print("="*60)

In [ ]:
# Get all family counts
family_counts = metadata['family'].value_counts()

# Create figure with scrollable y-axis
fig = go.Figure(go.Bar(
    x=family_counts.values,
    y=family_counts.index,
    orientation='h',
    marker=dict(
        color='#6366f1',
        line=dict(color='#4f46e5', width=0.5)
    ),
    text=family_counts.values,
    textposition='outside',
    textfont=dict(size=10),
    hovertemplate='<b>%{y}</b><br>Images: %{x}<br>Percentage: %{customdata:.2f}%<extra></extra>',
    customdata=(family_counts.values / len(metadata)) * 100
))

fig.update_layout(
    title={
        'text': '<b>Distribution Of Species By Family</b><br><sub>All families shown - scroll to explore</sub>',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 18}
    },
    xaxis_title='Number of Images',
    yaxis_title='Family',
    height=max(1000, len(family_counts) * 15),  # Dynamic height based on number of families
    width=1200,
    plot_bgcolor='white',
    paper_bgcolor='white',
    yaxis=dict(
        autorange='reversed',  # Highest count on top
        tickfont=dict(size=9),
        showgrid=False
    ),
    xaxis=dict(
        showgrid=True,
        gridcolor='#e5e7eb',
        gridwidth=1
    ),
    font=dict(size=11),
    margin=dict(l=200, r=100, t=100, b=50),  # More space for family names
    showlegend=False
)

fig.show()


In [ ]:

# Count at each taxonomic level
phylum_to_family = metadata.groupby(['phylum', 'family']).size().reset_index(name='count')

# Prepare data for Sankey
labels = list(metadata['phylum'].unique()) + list(metadata['family'].unique())
label_dict = {label: idx for idx, label in enumerate(labels)}

source = []
target = []
value = []

for _, row in phylum_to_family.iterrows():
    source.append(label_dict[row['phylum']])
    target.append(label_dict[row['family']])
    value.append(row['count'])

# Create Sankey diagram
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=labels,
        color='#6366f1'
    ),
    link=dict(
        source=source,
        target=target,
        value=value,
        color='rgba(99, 102, 241, 0.3)'
    )
)])

fig.update_layout(
    title='<b>Taxonomic Hierarchy: Phylum → Family</b>',
    font=dict(size=12),
    height=800,
    width=1200
)

fig.show()

print("\n📊 TAXONOMIC DIVERSITY:")
print(f"Phylums: {metadata['phylum'].nunique()}")
print(f"Families: {metadata['family'].nunique()}")
print(f"Average families per phylum: {metadata['family'].nunique() / metadata['phylum'].nunique():.1f}")

#### Retracting more information from the images

In [ ]:
# # Extract more information about the images and save to the metadata df (6min)
# metadata['image_size'], metadata['color_channel'], metadata['format'], metadata['aspect_ratio'], \
# metadata['min_val'], metadata['max_val'] = explore_image_files(directory + metadata['file_path'], explore_values=True)

# Don't explore values to save time (1min)
metadata['image_size'], metadata['color_channel'], metadata['format'], metadata['aspect_ratio'] = explore_image_files(directory + metadata['file_path'])

In [ ]:
metadata['width'], metadata['height'] = zip(*metadata['image_size'])

In [ ]:
metadata.describe()

huge file at "\rare_species\mollusca_cardiidae\30003931_46473744_eol-full-size-copy.jpg"

In [ ]:
# Show largest and smallest files
image_files = []
for root, dirs, files in os.walk(directory):
    for file in files:
        if file.lower().endswith(('.jpg', '.jpeg', '.png')):
            filepath = os.path.join(root, file)
            try:
                img = PilImage.open(filepath)
                pixel_count = img.size[0] * img.size[1]  # width * height
                image_files.append((filepath, pixel_count, img.size))
                img.close()
            except:
                pass

# Sort by pixel count
if image_files:
    image_files.sort(key=lambda x: x[1], reverse=True)
    largest_image_path, largest_pixels, largest_dims = image_files[0]
    smallest_image_path, smallest_pixels, smallest_dims = image_files[-1]
    
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    
    # Display largest image
    img_largest = PilImage.open(largest_image_path)
    axes[0].imshow(img_largest)
    axes[0].axis('off')
    axes[0].set_title(f"LARGEST IMAGE\n{os.path.basename(largest_image_path)}\n{largest_dims[0]} x {largest_dims[1]} pixels\n({largest_pixels:,} total pixels)", fontsize=12, fontweight='bold')
    
    # Display smallest image
    img_smallest = PilImage.open(smallest_image_path)
    axes[1].imshow(img_smallest)
    axes[1].axis('off')
    axes[1].set_title(f"SMALLEST IMAGE\n{os.path.basename(smallest_image_path)}\n{smallest_dims[0]} x {smallest_dims[1]} pixels\n({smallest_pixels:,} total pixels)", fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print(f"Largest:  {largest_image_path}")
    print(f"          {largest_dims[0]} x {largest_dims[1]} = {largest_pixels:,} pixels")
    print(f"\nSmallest: {smallest_image_path}")
    print(f"          {smallest_dims[0]} x {smallest_dims[1]} = {smallest_pixels:,} pixels")
else:
    print("Error")

- We see a normal looking smallest image, but the largest image looks like it could be split into 4 

## Outlier detection

### Outlier detection using YOLO

In [ ]:
pip install ultralytics

In [ ]:


# This will automatically download the model if it doesn't exist
model = YOLOWorld('yolov8s.pt') 

In [ ]:
from ultralytics import YOLO
import os
from PIL import Image as PilImage

MODEL_ID = 'yolov8s.pt'
CONFIDENCE_THRESHOLD = 0.85
PERSON_CLASS_ID = 0

def find_images_with_people(directory='rare_species/', 
                            conf_threshold=CONFIDENCE_THRESHOLD):
    print(f"Loading {MODEL_ID}...")
    model = YOLO(MODEL_ID)
    
    candidates = []
    file_count = 0
    
    print(f"Scanning for people with confidence > {conf_threshold}...")
    
    for root, _, files in os.walk(directory):
        for file in files:
            if file.lower().endswith(('.jpg', '.jpeg', '.png')):
                filepath = os.path.join(root, file)
                file_count += 1
                
                if file_count % 500 == 0:
                    print(f"  Processed {file_count} images...")
                
                try:
                    # Run prediction WITH the confidence threshold
                    results = model.predict(
                        filepath, 
                        verbose=False, 
                        conf=conf_threshold,  # This filters during prediction
                        iou=0.5,
                        classes=[PERSON_CLASS_ID]
                    )
                    
                    boxes = results[0].boxes if results and results[0].boxes is not None else None
                    
                    # Only add if there are detections AFTER filtering
                    if boxes and len(boxes) > 0:
                        img = PilImage.open(filepath)
                        candidates.append({
                            'path': filepath,
                            'dimensions': img.size,
                            'total_subjects': len(boxes),
                            'result': results[0]
                        })
                        img.close()
                
                except Exception as e:
                    pass

    print(f"\nDone. Processed {file_count} images. Found {len(candidates)} with people.")
    return candidates

In [ ]:
# Execute Step 1: Find the candidates
candidates = find_multiple_subject_images_zero_shot(directory=directory)

if candidates:
    print(f"\nTop 5 Candidates for visualization (sorted by subject count):")
    for i, c in enumerate(candidates[:5]):
        print(f"  {i+1}. {os.path.basename(c['path'])} - Subjects Detected: {c['total_subjects']}")
else:
    print("\nNo multi-subject candidates found based on the current settings.")

In [ ]:
def visualize_candidates(candidate_list, num_examples=10):
    """Loads the original image and plots the detection results directly onto it."""
    
    if not candidate_list:
        print("No candidates to visualize.")
        return

    print(f"\nDisplaying top {min(num_examples, len(candidate_list))} suggested images:")
    
    for i, item in enumerate(candidate_list[:num_examples]):
        try:
            # The 'result' object contains the image and detection data
            plot_image_array = item['result'].plot() 
            
            # Convert the NumPy array (BGR format from OpenCV) to a PIL Image (RGB)
            img_rgb = cv2.cvtColor(plot_image_array, cv2.COLOR_BGR2RGB)
            pil_img = PilImage.fromarray(img_rgb)
            
            print(f"\n--- Example {i+1}: {os.path.basename(item['path'])} ({item['total_subjects']} subjects) ---")
            display(pil_img) # Display the annotated image in the notebook
            
        except Exception as e:
            print(f"Could not visualize {os.path.basename(item['path'])}. Error: {e}")

# Execute Step 2: Visualize the top 3 results
visualize_candidates(candidates, num_examples=10)

# If the results look good, you can proceed with the cropping function 
# provided in the previous turn, or adapt it for more selective cropping.

In [ ]:
import random

def visualize_random_candidates(candidate_list, num_examples):
    """
    Selects a random sample of candidates and displays the annotated images.
    """
    if not candidate_list:
        print("The candidate list is empty. No images to visualize.")
        return

    # Determine how many examples to sample
    sample_size = min(num_examples, len(candidate_list))
    
    # Randomly select candidates
    random_sample = random.sample(candidate_list, sample_size)
    
    print(f"\n--- Displaying {sample_size} Random Suggested Images ---")
    
    # Reuse the logic from the previous visualization cell
    for i, item in enumerate(random_sample):
        try:
            # The 'result' object contains the image and detection data
            plot_image_array = item['result'].plot() 
            
            # Convert the NumPy array (BGR format from OpenCV) to a PIL Image (RGB)
            img_rgb = cv2.cvtColor(plot_image_array, cv2.COLOR_BGR2RGB)
            pil_img = PilImage.fromarray(img_rgb)
            
            print(f"\nExample {i+1} (Random Sample): {os.path.basename(item['path'])} ({item['total_subjects']} subjects)")
            display(pil_img) # Display the annotated image
            
        except Exception as e:
            print(f"Could not visualize random image {os.path.basename(item['path'])}. Error: {e}")

# Run the random sampling visualization
visualize_random_candidates(candidates, num_examples=20)

### Outlier detection using CLIP

In [ ]:
pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu

In [ ]:
from transformers import CLIPProcessor, CLIPModel
import torch

# Check for GPU/MPS availability
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"  # Apple Silicon
else:
    device = "cpu"
print(f"Using device: {device}")

# Load CLIP model
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# Define semantic categories - "good" images first, then "bad" noise types
text_prompts = [
    # Good categories (what we WANT to keep)
    "a photograph of an animal",
    "a wildlife photograph",
    "a photo of an animal in nature",
    
    # Bad categories (what we WANT to filter out)
    "an x-ray image",
    "a medical scan",
    "a drawing or sketch",
    "a map or diagram",
    "text document or book page",
    "a logo or icon",
    "a cooked meal",
    "prepared food",
    "A human",

    # Human-related (expanded)
    "a photo of a person",
    "a portrait of a human face",
    "people in a photograph",
    "a group of people",
    "a scientist or researcher",  # common in wildlife datasets    
]

# Precompute text embeddings (only need to do this once)
text_inputs = clip_processor(text=text_prompts, return_tensors="pt", padding=True).to(device)
with torch.no_grad():
    text_features = clip_model.get_text_features(**text_inputs)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)  # Normalize

In [ ]:
def get_clip_scores(image_path, directory="rare_species/"):
    """
    Get CLIP similarity scores for an image against all text prompts.
    Returns dict with scores and the predicted category.
    """
    try:
        # Load and process image
        full_path = directory + image_path if not image_path.startswith(directory) else image_path
        image = PilImage.open(full_path).convert("RGB")
        
        # Process image through CLIP
        image_inputs = clip_processor(images=image, return_tensors="pt").to(device)
        
        with torch.no_grad():
            image_features = clip_model.get_image_features(**image_inputs)
            image_features = image_features / image_features.norm(dim=-1, keepdim=True)
            
            # Calculate similarity with all text prompts
            similarities = (image_features @ text_features.T).squeeze().cpu().numpy()
        
        # Create results dict
        results = {
            'scores': dict(zip(text_prompts, similarities)),
            'best_match': text_prompts[similarities.argmax()],
            'best_score': similarities.max(),
            # Score for being a "good" photo (max of first 3 prompts)
            'photo_score': similarities[:3].max(),
            # Score for being "noise" (max of remaining prompts)  
            'noise_score': similarities[3:].max(),
        }
        # Semantic quality = how much more it looks like a photo vs noise
        results['semantic_quality'] = results['photo_score'] - results['noise_score']
        
        return results
        
    except Exception as e:
        print(f"Error processing {image_path}: {e}")
        return None

In [ ]:
def compute_clip_scores_batch(metadata, directory="rare_species/", batch_size=32):
    """
    Process all images in metadata and add CLIP scores.
    Uses batching for efficiency.
    """
    all_results = []
    file_paths = metadata['file_path'].tolist()
    
    for i in tqdm(range(0, len(file_paths), batch_size), desc="Computing CLIP scores"):
        batch_paths = file_paths[i:i+batch_size]
        batch_images = []
        valid_indices = []
        
        # Load batch of images
        for j, path in enumerate(batch_paths):
            try:
                full_path = directory + path
                img = PilImage.open(full_path).convert("RGB")
                batch_images.append(img)
                valid_indices.append(i + j)
            except Exception as e:
                print(f"Error loading {path}: {e}")
                all_results.append({
                    'clip_photo_score': None,
                    'clip_noise_score': None,
                    'clip_semantic_quality': None,
                    'clip_best_match': None
                })
        
        if not batch_images:
            continue
            
        # Process batch through CLIP
        image_inputs = clip_processor(images=batch_images, return_tensors="pt", padding=True).to(device)
        
        with torch.no_grad():
            image_features = clip_model.get_image_features(**image_inputs)
            image_features = image_features / image_features.norm(dim=-1, keepdim=True)
            similarities = (image_features @ text_features.T).cpu().numpy()
        
        # Extract scores for each image in batch
        for j, sims in enumerate(similarities):
            photo_score = sims[:3].max()
            noise_score = sims[3:].max()
            all_results.append({
                'clip_photo_score': float(photo_score),
                'clip_noise_score': float(noise_score),
                'clip_semantic_quality': float(photo_score - noise_score),
                'clip_best_match': text_prompts[sims.argmax()]
            })
    
    return pd.DataFrame(all_results)

In [ ]:
# Run CLIP scoring on your metadata
print("Computing CLIP semantic scores...")
clip_results = compute_clip_scores_batch(metadata, directory=directory, batch_size=32)

# Add CLIP columns to metadata
metadata['clip_photo_score'] = clip_results['clip_photo_score']
metadata['clip_noise_score'] = clip_results['clip_noise_score']
metadata['clip_semantic_quality'] = clip_results['clip_semantic_quality']
metadata['clip_best_match'] = clip_results['clip_best_match']

print(f"Processed {len(metadata)} images")

In [ ]:
# Show images CLIP classified as noise/outliers
def show_clip_bad_images(metadata, directory="rare_species/", n_cols=8, n_rows=6):
    """Display images that CLIP thinks are NOT real animal photographs."""
    
    # Filter to bad images (noise score > photo score)
    bad_images = metadata[metadata['clip_semantic_quality'] < 0].copy()
    bad_images = bad_images.sort_values('clip_semantic_quality', ascending=True)  # Worst first
    
    if len(bad_images) == 0:
        print("No bad images detected by CLIP!")
        return
    
    print(f"Found {len(bad_images)} images CLIP thinks are NOT real animal photos\n")
    print("Category breakdown:")
    print(bad_images['clip_best_match'].value_counts())
    print("\n" + "="*60 + "\n")
    
    # Show grid of worst offenders
    n_show = min(n_cols * n_rows, len(bad_images))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.5*n_cols, 3*n_rows))
    axes = axes.flatten()
    
    for idx, (_, row) in enumerate(bad_images.head(n_show).iterrows()):
        try:
            img = PilImage.open(directory + row['file_path'])
            axes[idx].imshow(img)
            axes[idx].set_title(
                f"{row['clip_best_match'].split()[-1]}\n"  # Short category name
                f"score: {row['clip_semantic_quality']:.2f}",
                fontsize=8
            )
        except Exception as e:
            axes[idx].set_title(f"Error: {e}", fontsize=8)
        axes[idx].axis('off')
    
    # Hide unused subplots
    for idx in range(n_show, len(axes)):
        axes[idx].axis('off')
    
    plt.suptitle(f"CLIP-Detected Outliers (Worst {n_show} of {len(bad_images)})", 
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Run it
show_clip_bad_images(metadata, directory=directory)

In [ ]:
def show_all_outliers(metadata, directory="rare_species/", n_cols=10):
    """Display ALL CLIP outliers."""
    
    bad_images = metadata[metadata['clip_semantic_quality'] < 0].copy()
    bad_images = bad_images.sort_values('clip_semantic_quality', ascending=True)
    
    if len(bad_images) == 0:
        print("No outliers found!")
        return
    
    print(f"Showing ALL {len(bad_images)} outliers\n")
    print("Category breakdown:")
    print(bad_images['clip_best_match'].value_counts())
    print("\n" + "="*60 + "\n")
    
    n_rows = math.ceil(len(bad_images) / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(2*n_cols, 2.5*n_rows))
    axes = axes.flatten()
    
    for idx, (_, row) in enumerate(bad_images.iterrows()):
        try:
            img = PilImage.open(directory + row['file_path'])
            axes[idx].imshow(img)
            axes[idx].set_title(
                f"{row['clip_best_match'].replace('a ', '').replace('an ', '')[:15]}\n"
                f"{row['clip_semantic_quality']:.2f}",
                fontsize=6
            )
        except:
            pass
        axes[idx].axis('off')
    
    for idx in range(len(bad_images), len(axes)):
        axes[idx].axis('off')
    
    plt.suptitle(f"All Outliers ({len(bad_images)} images)", 
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

show_all_outliers(metadata, directory=directory)

## Colour exploration

In [ ]:
# Plot colour channel distribution
color_channel_mapping = {
    'L': 'Greyscale',
    'RGB': 'RGB',
    'RGBA': 'RGBA',
    'P': 'Palette',
    'CMYK': 'CMYK',
    '1': 'Binary',
    'LA': 'Greyscale + Alpha'
}

# Get value counts and map to readable names
color_counts = metadata['color_channel'].value_counts()
color_counts.index = color_counts.index.map(lambda x: color_channel_mapping.get(x, x))

# Create figure
fig = go.Figure(go.Bar(
    x=color_counts.index,
    y=color_counts.values,
    marker=dict(
        color='#6366f1',
        line=dict(color='#4f46e5', width=0.5)
    ),
    text=color_counts.values,
    textposition='outside',
    textfont=dict(size=12),
    hovertemplate='<b>%{x}</b><br>Images: %{y}<br>Percentage: %{customdata:.2f}%<extra></extra>',
    customdata=(color_counts.values / len(metadata)) * 100
))

fig.update_layout(
    title={
        'text': '<b>Distribution Of Colour Channels</b>',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 18}
    },
    xaxis_title='Color Channel',
    yaxis_title='Count',
    height=600,
    width=1000,
    plot_bgcolor='white',
    paper_bgcolor='white',
    xaxis=dict(
        tickfont=dict(size=11),
        showgrid=False
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor='#e5e7eb',
        gridwidth=1
    ),
    font=dict(size=11),
    showlegend=False
)

fig.show()

In [ ]:
# Custom lines based on previous discoveries
common_ratio = 4/3
width_max = 500        # MB CHANGE LATER

In [ ]:
# Create figure with custom styling
fig = go.Figure()

# Add scatter plot with better styling
fig.add_trace(go.Scatter(
    x=metadata['width'],
    y=metadata['height'],
    mode='markers',
    marker=dict(
        size=4,
        color='#6366f1',
        opacity=0.4,
        line=dict(width=0)
    ),
    name='Images',
    hovertemplate='<b>Width:</b> %{x}px<br><b>Height:</b> %{y}px<extra></extra>'
))

# Add aspect ratio reference line (4:3)
max_y = metadata['height'].max()
max_x_for_ratio = common_ratio * max_y
fig.add_trace(go.Scatter(
    x=[0, max_x_for_ratio],
    y=[0, max_y],
    mode='lines',
    line=dict(color='#ef4444', width=2.5, dash='dash'),
    name='4:3 Aspect Ratio',
    hoverinfo='skip'
))

# Add width threshold line
fig.add_trace(go.Scatter(
    x=[width_max, width_max],
    y=[0, max_y],
    mode='lines',
    line=dict(color='#10b981', width=2.5, dash='dash'),
    name=f'Max Width ({width_max}px)',
    hoverinfo='skip'
))

# Update layout with legend on the right and bold title
fig.update_layout(
    title={
        'text': '<b>Image Size Distribution Analysis</b>',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 22, 'color': '#1f2937'}
    },
    xaxis=dict(
        title='Width (pixels)',
        showgrid=True,
        gridcolor='#e5e7eb',
        gridwidth=1,
        zeroline=False,
        title_font=dict(size=14, color='#374151')
    ),
    yaxis=dict(
        title='Height (pixels)',
        showgrid=True,
        gridcolor='#e5e7eb',
        gridwidth=1,
        zeroline=False,
        title_font=dict(size=14, color='#374151')
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=1000,
    height=700,
    showlegend=True,
    legend=dict(
        x=1.02,
        y=1,
        xanchor='left',
        yanchor='top',
        bgcolor='rgba(255, 255, 255, 0.95)',
        bordercolor='#d1d5db',
        borderwidth=1,
        font=dict(size=11)
    ),
    hovermode='closest'
)

fig.show()

# Enhanced summary statistics
image_size = (width_max, round(width_max/common_ratio))
print(f'\n{"="*50}')
print(f'📊 RECOMMENDED IMAGE SIZE: {image_size[0]}x{image_size[1]}')
print(f'{"="*50}')
print(f'   Aspect Ratio: {common_ratio:.2f}:1 (4:3)')
print(f'   Total Images: {len(metadata):,}')
print(f'   Width Range: {metadata["width"].min()}-{metadata["width"].max()}px')
print(f'   Height Range: {metadata["height"].min()}-{metadata["height"].max()}px')
print(f'{"="*50}\n')

In [ ]:
# Create 2D histogram/heatmap of image dimensions
fig = px.density_heatmap(
    metadata,
    x='width',
    y='height',
    nbinsx=50,
    nbinsy=50,
    title='<b>Image Dimension Density Heatmap</b>',
    labels={'width': 'Width (pixels)', 'height': 'Height (pixels)'},
    color_continuous_scale='Blues'
)

fig.update_layout(
    width=900,
    height=700,
    plot_bgcolor='white'
)

fig.show()

### Images exploration

#### All images

In [ ]:
# Print 5 example images of each class (+-3min)
for label in os.listdir(directory):
    path = directory + str(label)

    if not os.path.isdir(path):
        print(f"Directory {path} does not exist.")
        continue

    folder_data = os.listdir(path)
    k = 0
    print(f'{label} ({len(folder_data)} images)')

    # Collect image paths
    image_paths = []
    for image_path in folder_data:
        if k < 5:                                               # <-- change how many images per class
            full_path = os.path.join(path, image_path)
            image_paths.append(full_path)
            k += 1

    # Display images
    if image_paths:
        fig, axes = plt.subplots(1, len(image_paths), figsize=(15, 3))
        if len(image_paths) == 1:
            axes = [axes]

        for ax, img_path in zip(axes, image_paths):
            img = PilImage.open(img_path)
            ax.imshow(img)
            ax.axis('off')

        plt.tight_layout()
        plt.show()

From printing some examples of the images that we'll be working with we can see that we have a few:

- X-ray imagaes
- Images of signs
- Paintings
- Text extracts with no images
- Images unrelated to the class
- Maps
- Varying zoom / color / rotations

#### Grayscale VS CMYK

In [ ]:
datagen = ImageDataGenerator(rescale=1./255)

# Get only greyscale images
temp_generator = datagen.flow_from_dataframe(
    dataframe=metadata[metadata.color_channel == 'L'],
    directory=directory,
    x_col='file_path',
    y_col='family',
    target_size=image_size,
    batch_size=32,
    class_mode='categorical',
    shuffle=False   # Not shuffling
)

In [ ]:
# Get class names from the generator
class_names = list(temp_generator.class_indices.keys())

# Determine the total number of batches in the generator
num_batches = int(math.ceil(temp_generator.n / temp_generator.batch_size))

for i in range(num_batches):
    images, labels = next(temp_generator)

    # Plot the images in the current batch
    batch_size_actual = images.shape[0]
    n_cols = min(8, batch_size_actual)                  # <-- set max columns per row
    n_rows = math.ceil(batch_size_actual / n_cols)

    plt.figure(figsize=(3 * n_cols, 3 * n_rows))

    for j in range(batch_size_actual):
        ax = plt.subplot(n_rows, n_cols, j + 1)
        plt.imshow(images[j])

        # Get the index of the highest probability to find the class name
        label_idx = np.argmax(labels[j])
        plt.title(class_names[label_idx], fontsize=9)
        plt.axis("off")

    plt.tight_layout()
    plt.show()

Some images retain at least the shape of the actual animal, but most of them are really bad

In [ ]:
# Do the same for CMYK
temp_generator = datagen.flow_from_dataframe(
    dataframe=metadata[metadata.color_channel == 'CMYK'],
    directory=directory,
    x_col='file_path',
    y_col='family',
    target_size=image_size,
    batch_size=32,
    class_mode='categorical',
    shuffle=False   # Not shuffling
)

In [ ]:
# Get class names from the generator
class_names = list(temp_generator.class_indices.keys())

# Determine the total number of batches in the generator
num_batches = int(math.ceil(temp_generator.n / temp_generator.batch_size))

for i in range(num_batches):
    images, labels = next(temp_generator)

    # Plot the images in the current batch
    batch_size_actual = images.shape[0]
    n_cols = min(8, batch_size_actual)                  # <-- set max columns per row
    n_rows = math.ceil(batch_size_actual / n_cols)

    plt.figure(figsize=(3 * n_cols, 3 * n_rows))

    for j in range(batch_size_actual):
        ax = plt.subplot(n_rows, n_cols, j + 1)
        plt.imshow(images[j])

        # Get the index of the highest probability to find the class name
        label_idx = np.argmax(labels[j])
        plt.title(class_names[label_idx], fontsize=9)
        plt.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
# Changing the cmyk outliers to rgb in the metadata

metadata.loc[metadata['family'].isin(['chordata_emydidae', 'chordata_psittacidae']) & 
             (metadata['color_channel'] == 'CMYK'), 'color_channel'] = 'RGB'

Most of the images apper to be x-rays of sculls of different animals which don't really help identify the animals by the picture, and will more likely only confuse the model. \
(and there are 2 different images for some reason: 2 parrots and a turtle)

In [ ]:
# Delete the generator to free up resources (small impactbut why not)
del temp_generator

## Preprocessing

In [ ]:
metadata = metadata[~metadata['color_channel'].isin(['CMYK', 'L'])].reset_index(drop=True)

print(f"Removed CMYK and grayscale images. Remaining: {len(metadata)}")

In [ ]:
# TRY TO ONLY USE RGB
# metadata = metadata[metadata.color_channel == 'RGB'].reset_index(drop=True)

In [ ]:
# good_images_df = good_images_df[good_images_df.family != 'formicidae'].reset_index(drop=True)

## Outlier detection / Removal

### Outlier detection using YOLO

In [ ]:
pip install ultralytics

In [ ]:


# This will automatically download the model if it doesn't exist
model = YOLOWorld('yolov8s.pt') 

In [ ]:
from ultralytics import YOLO
import os
from PIL import Image as PilImage

MODEL_ID = 'yolov8s.pt'
CONFIDENCE_THRESHOLD = 0.85
PERSON_CLASS_ID = 0

def find_images_with_people(directory='rare_species/', 
                            conf_threshold=CONFIDENCE_THRESHOLD):
    print(f"Loading {MODEL_ID}...")
    model = YOLO(MODEL_ID)
    
    candidates = []
    file_count = 0
    
    print(f"Scanning for people with confidence > {conf_threshold}...")
    
    for root, _, files in os.walk(directory):
        for file in files:
            if file.lower().endswith(('.jpg', '.jpeg', '.png')):
                filepath = os.path.join(root, file)
                file_count += 1
                
                if file_count % 500 == 0:
                    print(f"  Processed {file_count} images...")
                
                try:
                    # Run prediction WITH the confidence threshold
                    results = model.predict(
                        filepath, 
                        verbose=False, 
                        conf=conf_threshold,  # This filters during prediction
                        iou=0.5,
                        classes=[PERSON_CLASS_ID]
                    )
                    
                    boxes = results[0].boxes if results and results[0].boxes is not None else None
                    
                    # Only add if there are detections AFTER filtering
                    if boxes and len(boxes) > 0:
                        img = PilImage.open(filepath)
                        candidates.append({
                            'path': filepath,
                            'dimensions': img.size,
                            'total_subjects': len(boxes),
                            'result': results[0]
                        })
                        img.close()
                
                except Exception as e:
                    pass

    print(f"\nDone. Processed {file_count} images. Found {len(candidates)} with people.")
    return candidates

In [ ]:
# Execute Step 1: Find the candidates
candidates = find_multiple_subject_images_zero_shot(directory=directory)

if candidates:
    print(f"\nTop 5 Candidates for visualization (sorted by subject count):")
    for i, c in enumerate(candidates[:5]):
        print(f"  {i+1}. {os.path.basename(c['path'])} - Subjects Detected: {c['total_subjects']}")
else:
    print("\nNo multi-subject candidates found based on the current settings.")

In [ ]:
def visualize_candidates(candidate_list, num_examples=10):
    """Loads the original image and plots the detection results directly onto it."""
    
    if not candidate_list:
        print("No candidates to visualize.")
        return

    print(f"\nDisplaying top {min(num_examples, len(candidate_list))} suggested images:")
    
    for i, item in enumerate(candidate_list[:num_examples]):
        try:
            # The 'result' object contains the image and detection data
            plot_image_array = item['result'].plot() 
            
            # Convert the NumPy array (BGR format from OpenCV) to a PIL Image (RGB)
            img_rgb = cv2.cvtColor(plot_image_array, cv2.COLOR_BGR2RGB)
            pil_img = PilImage.fromarray(img_rgb)
            
            print(f"\n--- Example {i+1}: {os.path.basename(item['path'])} ({item['total_subjects']} subjects) ---")
            display(pil_img) # Display the annotated image in the notebook
            
        except Exception as e:
            print(f"Could not visualize {os.path.basename(item['path'])}. Error: {e}")

# Execute Step 2: Visualize the top 3 results
visualize_candidates(candidates, num_examples=10)

# If the results look good, you can proceed with the cropping function 
# provided in the previous turn, or adapt it for more selective cropping.

In [ ]:
import random

def visualize_random_candidates(candidate_list, num_examples):
    """
    Selects a random sample of candidates and displays the annotated images.
    """
    if not candidate_list:
        print("The candidate list is empty. No images to visualize.")
        return

    # Determine how many examples to sample
    sample_size = min(num_examples, len(candidate_list))
    
    # Randomly select candidates
    random_sample = random.sample(candidate_list, sample_size)
    
    print(f"\n--- Displaying {sample_size} Random Suggested Images ---")
    
    # Reuse the logic from the previous visualization cell
    for i, item in enumerate(random_sample):
        try:
            # The 'result' object contains the image and detection data
            plot_image_array = item['result'].plot() 
            
            # Convert the NumPy array (BGR format from OpenCV) to a PIL Image (RGB)
            img_rgb = cv2.cvtColor(plot_image_array, cv2.COLOR_BGR2RGB)
            pil_img = PilImage.fromarray(img_rgb)
            
            print(f"\nExample {i+1} (Random Sample): {os.path.basename(item['path'])} ({item['total_subjects']} subjects)")
            display(pil_img) # Display the annotated image
            
        except Exception as e:
            print(f"Could not visualize random image {os.path.basename(item['path'])}. Error: {e}")

# Run the random sampling visualization
visualize_random_candidates(candidates, num_examples=20)

### Outlier detection using CLIP

In [ ]:
pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu

In [ ]:
from transformers import CLIPProcessor, CLIPModel
import torch

# Check for GPU/MPS availability
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"  # Apple Silicon
else:
    device = "cpu"
print(f"Using device: {device}")

# Load CLIP model
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# Define semantic categories - "good" images first, then "bad" noise types
text_prompts = [
    # Good categories (what we WANT to keep)
    "a photograph of an animal",
    "a wildlife photograph",
    "a photo of an animal in nature",
    
    # Bad categories (what we WANT to filter out)
    "an x-ray image",
    "a medical scan",
    "a drawing or sketch",
    "a map or diagram",
    "text document or book page",
    "a logo or icon",
    "a cooked meal",
    "prepared food",
    "A human",

    # Human-related (expanded)
    "a photo of a person",
    "a portrait of a human face",
    "people in a photograph",
    "a group of people",
    "a scientist or researcher",  # common in wildlife datasets    
]

# Precompute text embeddings (only need to do this once)
text_inputs = clip_processor(text=text_prompts, return_tensors="pt", padding=True).to(device)
with torch.no_grad():
    text_features = clip_model.get_text_features(**text_inputs)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)  # Normalize

In [ ]:
def get_clip_scores(image_path, directory="rare_species/"):
    """
    Get CLIP similarity scores for an image against all text prompts.
    Returns dict with scores and the predicted category.
    """
    try:
        # Load and process image
        full_path = directory + image_path if not image_path.startswith(directory) else image_path
        image = PilImage.open(full_path).convert("RGB")
        
        # Process image through CLIP
        image_inputs = clip_processor(images=image, return_tensors="pt").to(device)
        
        with torch.no_grad():
            image_features = clip_model.get_image_features(**image_inputs)
            image_features = image_features / image_features.norm(dim=-1, keepdim=True)
            
            # Calculate similarity with all text prompts
            similarities = (image_features @ text_features.T).squeeze().cpu().numpy()
        
        # Create results dict
        results = {
            'scores': dict(zip(text_prompts, similarities)),
            'best_match': text_prompts[similarities.argmax()],
            'best_score': similarities.max(),
            # Score for being a "good" photo (max of first 3 prompts)
            'photo_score': similarities[:3].max(),
            # Score for being "noise" (max of remaining prompts)  
            'noise_score': similarities[3:].max(),
        }
        # Semantic quality = how much more it looks like a photo vs noise
        results['semantic_quality'] = results['photo_score'] - results['noise_score']
        
        return results
        
    except Exception as e:
        print(f"Error processing {image_path}: {e}")
        return None

In [ ]:
def compute_clip_scores_batch(metadata, directory="rare_species/", batch_size=32):
    """
    Process all images in metadata and add CLIP scores.
    Uses batching for efficiency.
    """
    all_results = []
    file_paths = metadata['file_path'].tolist()
    
    for i in tqdm(range(0, len(file_paths), batch_size), desc="Computing CLIP scores"):
        batch_paths = file_paths[i:i+batch_size]
        batch_images = []
        valid_indices = []
        
        # Load batch of images
        for j, path in enumerate(batch_paths):
            try:
                full_path = directory + path
                img = PilImage.open(full_path).convert("RGB")
                batch_images.append(img)
                valid_indices.append(i + j)
            except Exception as e:
                print(f"Error loading {path}: {e}")
                all_results.append({
                    'clip_photo_score': None,
                    'clip_noise_score': None,
                    'clip_semantic_quality': None,
                    'clip_best_match': None
                })
        
        if not batch_images:
            continue
            
        # Process batch through CLIP
        image_inputs = clip_processor(images=batch_images, return_tensors="pt", padding=True).to(device)
        
        with torch.no_grad():
            image_features = clip_model.get_image_features(**image_inputs)
            image_features = image_features / image_features.norm(dim=-1, keepdim=True)
            similarities = (image_features @ text_features.T).cpu().numpy()
        
        # Extract scores for each image in batch
        for j, sims in enumerate(similarities):
            photo_score = sims[:3].max()
            noise_score = sims[3:].max()
            all_results.append({
                'clip_photo_score': float(photo_score),
                'clip_noise_score': float(noise_score),
                'clip_semantic_quality': float(photo_score - noise_score),
                'clip_best_match': text_prompts[sims.argmax()]
            })
    
    return pd.DataFrame(all_results)

In [ ]:
# Run CLIP scoring on your metadata
print("Computing CLIP semantic scores...")
clip_results = compute_clip_scores_batch(metadata, directory=directory, batch_size=32)

# Add CLIP columns to metadata
metadata['clip_photo_score'] = clip_results['clip_photo_score']
metadata['clip_noise_score'] = clip_results['clip_noise_score']
metadata['clip_semantic_quality'] = clip_results['clip_semantic_quality']
metadata['clip_best_match'] = clip_results['clip_best_match']

print(f"Processed {len(metadata)} images")

In [ ]:
# Show images CLIP classified as noise/outliers
def show_clip_bad_images(metadata, directory="rare_species/", n_cols=8, n_rows=6):
    """Display images that CLIP thinks are NOT real animal photographs."""
    
    # Filter to bad images (noise score > photo score)
    bad_images = metadata[metadata['clip_semantic_quality'] < 0].copy()
    bad_images = bad_images.sort_values('clip_semantic_quality', ascending=True)  # Worst first
    
    if len(bad_images) == 0:
        print("No bad images detected by CLIP!")
        return
    
    print(f"Found {len(bad_images)} images CLIP thinks are NOT real animal photos\n")
    print("Category breakdown:")
    print(bad_images['clip_best_match'].value_counts())
    print("\n" + "="*60 + "\n")
    
    # Show grid of worst offenders
    n_show = min(n_cols * n_rows, len(bad_images))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.5*n_cols, 3*n_rows))
    axes = axes.flatten()
    
    for idx, (_, row) in enumerate(bad_images.head(n_show).iterrows()):
        try:
            img = PilImage.open(directory + row['file_path'])
            axes[idx].imshow(img)
            axes[idx].set_title(
                f"{row['clip_best_match'].split()[-1]}\n"  # Short category name
                f"score: {row['clip_semantic_quality']:.2f}",
                fontsize=8
            )
        except Exception as e:
            axes[idx].set_title(f"Error: {e}", fontsize=8)
        axes[idx].axis('off')
    
    # Hide unused subplots
    for idx in range(n_show, len(axes)):
        axes[idx].axis('off')
    
    plt.suptitle(f"CLIP-Detected Outliers (Worst {n_show} of {len(bad_images)})", 
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Run it
show_clip_bad_images(metadata, directory=directory)

In [ ]:
def show_all_outliers(metadata, directory="rare_species/", n_cols=10):
    """Display ALL CLIP outliers."""
    
    bad_images = metadata[metadata['clip_semantic_quality'] < 0].copy()
    bad_images = bad_images.sort_values('clip_semantic_quality', ascending=True)
    
    if len(bad_images) == 0:
        print("No outliers found!")
        return
    
    print(f"Showing ALL {len(bad_images)} outliers\n")
    print("Category breakdown:")
    print(bad_images['clip_best_match'].value_counts())
    print("\n" + "="*60 + "\n")
    
    n_rows = math.ceil(len(bad_images) / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(2*n_cols, 2.5*n_rows))
    axes = axes.flatten()
    
    for idx, (_, row) in enumerate(bad_images.iterrows()):
        try:
            img = PilImage.open(directory + row['file_path'])
            axes[idx].imshow(img)
            axes[idx].set_title(
                f"{row['clip_best_match'].replace('a ', '').replace('an ', '')[:15]}\n"
                f"{row['clip_semantic_quality']:.2f}",
                fontsize=6
            )
        except:
            pass
        axes[idx].axis('off')
    
    for idx in range(len(bad_images), len(axes)):
        axes[idx].axis('off')
    
    plt.suptitle(f"All Outliers ({len(bad_images)} images)", 
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

show_all_outliers(metadata, directory=directory)

In [ ]:
# keep this imports here for how cuz we might change the model
from tensorflow.keras.applications import EfficientNetB4    
from tensorflow.keras.applications.efficientnet import preprocess_input

In [ ]:
# Load pre-trained EfficientNet without the top classification layer
base_model_embedding = EfficientNetB4(weights='imagenet', include_top=False, pooling='avg')

# Create a model that outputs embeddings
embedding_model = Model(inputs=base_model_embedding.input, outputs=base_model_embedding.output)
print(f"Embedding dimension: {base_model_embedding.output_shape[-1]}")

In [ ]:
# Create generator with preprocessing form the model
datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

# Create generator 
preprocessing_generator = datagen.flow_from_dataframe(
    dataframe=metadata,
    directory=directory,
    x_col='file_path',
    y_col='family',  
    target_size=image_size,
    batch_size=32,  
    class_mode='categorical',
    color_mode='rgb',
    shuffle=False       # Not to mess up the order with df
)


# Generate embeddings
embeddings_list = []

# Calculate number of batches
num_batches = len(preprocessing_generator)

for i in tqdm(range(num_batches)):  #(+-4.5min)
    batch_images, _ = next(preprocessing_generator)  # Get images (ignore labels)
    
    # Generate embeddings
    batch_embeddings = embedding_model.predict(batch_images, verbose=0) 
    embeddings_list.extend(batch_embeddings)

In [ ]:
# Kill the generator to free up resources (not actually sure that even helps)
del preprocessing_generator

In [ ]:
# Add embedings to metadata
metadata['embedding'] = embeddings_list

In [ ]:
def distance_to_centroid(metadata):
    # Calculate a centroid for each family
    centroids = {}

    for family in metadata['family'].unique():
        family_embeddings = metadata[
            (metadata['family'] == family) &
            (metadata['embedding'].notna())
        ]['embedding'].tolist()
        

        if len(family_embeddings) > 0:
            # Stack embeddings and calculate mean
            embeddings_array = np.stack(family_embeddings)
            centroids[family] = np.mean(embeddings_array, axis=0)
    

    # Calculate distances to centroids
    distances = []

    for idx, row in metadata.iterrows():
        if pd.isna(row['embedding'].any()):
            distances.append(np.nan)
        else:
            family = row['family']
            if family in centroids:
                # Calculate Euclidean distance
                distance = np.linalg.norm(row['embedding'] - centroids[family])
                distances.append(distance)
            else:
                distances.append(np.nan)
    
    return distances

In [ ]:
metadata['distance_to_centroid'] = distance_to_centroid(metadata)

In [ ]:
def show_outlier_images(metadata, N):
   # Get the images sorted by distance to centroid (make a copy to not mess up the original metadata)
    sorted_metadata = metadata.sort_values(by='distance_to_centroid', ascending=False).copy()

    # Display the first N images
    n_cols = 10
    n_rows = (N + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 2, n_rows * 2))
    # Flatten axes to simplify indexing
    if isinstance(axes, np.ndarray):
        axes = axes.flatten()
    else:
        axes = [axes]
        

    for idx in range(N):
        if idx < len(sorted_metadata):
            img_path = os.path.join(directory, sorted_metadata.iloc[idx]['file_path'])
            img = PilImage.open(img_path)
            axes[idx].imshow(img)
            family = sorted_metadata.iloc[idx]['family']
            axes[idx].set_title(f'{family} (id={idx})', fontsize=8)
            axes[idx].axis('off')

        else:
            # No image for this subplot, hide it
            axes[idx].axis('off')

    # Hide any remaining unused subplots (in case N < n_rows * n_cols)
    for j in range(N, n_rows * n_cols):
        axes[j].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
def show_outlier_images_batched(metadata, N, n_cols=10):
   # Get the images sorted by distance to centroid (make a copy to not mess up the original metadata)
    sorted_metadata = metadata.sort_values(by='distance_to_centroid', ascending=False).copy()

    for i in range(0, N, n_cols):
        fig, axes = plt.subplots(1, n_cols, figsize=(n_cols * 2, 2))
        # Flatten axes to simplify indexing
        if isinstance(axes, np.ndarray):
            axes = axes.flatten()
        else:
            axes = [axes]
            

        for idx in range(i, min(i + n_cols, N)):
            if idx < len(sorted_metadata):
                img_path = os.path.join(directory, sorted_metadata.iloc[idx]['file_path'])
                img = PilImage.open(img_path)
                axes[idx - i].imshow(img)
                family = sorted_metadata.iloc[idx]['family']
                axes[idx - i].set_title(f'{family} (id={idx})', fontsize=8)
                axes[idx - i].axis('off')

            else:
                # No image for this subplot, hide it
                axes[idx - i].axis('off')

        plt.tight_layout()
        plt.show()

In [ ]:
# show_outlier_images_batched(metadata, 1200)

In [ ]:
# Identify outliers based on distance to centroid
threshold = metadata['distance_to_centroid'].quantile(0.95) # top 5% as outliers
outliers = metadata[metadata['distance_to_centroid'] > threshold].copy()
good_images_df = metadata[metadata['distance_to_centroid'] < threshold].copy()

print(f"Found {len(outliers)} outliers")

In [ ]:
# Recalculate centroids
good_images_df.drop('distance_to_centroid', axis=1, inplace=True)
good_images_df['distance_to_centroid'] = distance_to_centroid(good_images_df)

In [ ]:
# show_outlier_images(good_images_df, 300)

In [ ]:
# Identify additional outliers based on updated distances
threshold = good_images_df['distance_to_centroid'].quantile(0.95)
outliers = pd.concat([outliers, good_images_df[good_images_df['distance_to_centroid'] > threshold]], ignore_index=True)
good_images_df = good_images_df[good_images_df['distance_to_centroid'] < threshold].copy()

print(f"Found {len(outliers)} outliers")

In [ ]:
low_images_class = good_images_df['family'].value_counts().idxmin()
print(f"Total images after outlier removal: {len(good_images_df)} (out of {len(metadata)})")
print(f"The lowest amount of images per class {good_images_df['family'].value_counts().iloc[-1]} in family '{low_images_class}'")

In [ ]:
# show_outlier_images_batched(metadata[metadata.family == 'cheloniidae'], 30)

In [ ]:
# 2 good images and then outliers
good_images = 2

for family in outliers['family'].unique():
    image_paths = list(directory+good_images_df[good_images_df.family == family].sort_values('distance_to_centroid')['file_path'])[:good_images]
    image_paths.extend(list(directory+outliers[outliers.family == family].sort_values('distance_to_centroid', ascending = False)['file_path']))

    if image_paths:
        # Set number of columns and calculate rows needed
        n_cols = 7
        n_images = len(image_paths)
        n_rows = (n_images + n_cols - 1) // n_cols  # Ceiling division

        # Create subplots with appropriate dimensions
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(9, 2 * n_rows))

        # Flatten axes array for easier iteration
        if n_rows == 1 and n_cols == 1:
            axes = [axes]
        else:
            axes = axes.flatten()

        # Display images
        for idx, img_path in enumerate(image_paths):
            img = PilImage.open(img_path).resize((224, 224), PilImage.LANCZOS)
            axes[idx].imshow(img)
            axes[idx].axis('off')

        # Add green outline for good images (first good_count images)
            if idx < good_images:
                rect = plt.Rectangle((0, 0), 223, 223, fill=False,
                                    edgecolor='lime', linewidth=3)
                axes[idx].add_patch(rect)

        # Hide any unused subplots
        for idx in range(n_images, n_rows * n_cols):
            axes[idx].axis('off')

        plt.suptitle(f"{family}, {good_images_df[good_images_df.family == family].shape[0]} good images left")#, fontsize=16, y=0.9, x=0.1)
        plt.tight_layout()
        plt.show()

### Imagenet outlier detection

In [ ]:
# Load InceptionV3 model pretrained on ImageNet
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.applications.inception_v3 import preprocess_input, decode_predictions

print("Loading InceptionV3 model...")
imagenet_model = InceptionV3(weights='imagenet')
print("Model loaded successfully")

In [ ]:
def check_animal_presence(img_path, model, threshold=0.5):
    """
    Check if image contains an animal using ImageNet classifications
    
    Args:
        img_path: Path to image
        model: InceptionV3 model
        threshold: Minimum probability threshold
    
    Returns:
        is_animal: bool
        max_animal_prob: float (probability of top animal class)
        top_class: str (name of top predicted class)
        top_prob: float (probability of top class)
    """
    try:
        # Load and preprocess
        img = tf.keras.preprocessing.image.load_img(img_path, target_size=(299, 299))
        img_array = tf.keras.preprocessing.image.img_to_array(img)
        img_array = preprocess_input(img_array)
        img_array = np.expand_dims(img_array, axis=0)
        
        # Predict
        preds = model.predict(img_array, verbose=0)
        
        # Get animal class probabilities (classes 0-397 are mostly animals in ImageNet)
        animal_probs = preds[0, :398]
        max_animal_prob = np.max(animal_probs)
        
        # Decode top prediction
        top_preds = decode_predictions(preds, top=1)[0]
        top_class = top_preds[0][1]
        top_prob = top_preds[0][2]
        
        is_animal = max_animal_prob > threshold
        
        return is_animal, max_animal_prob, top_class, top_prob
    
    except Exception as e:
        print(f"Error processing {img_path}: {e}")
        return False, 0.0, "error", 0.0

In [ ]:
is_animal_list = []
animal_prob_list = []
top_class_list = []
top_prob_list = []

for idx, row in tqdm(metadata.iterrows(), total=len(metadata), desc="Detecting animals"):
    img_path = os.path.join(directory, row['file_path'])
    is_animal, animal_prob, top_class, top_prob = check_animal_presence(
        img_path, imagenet_model, threshold=0.02
    )
    is_animal_list.append(is_animal)
    animal_prob_list.append(animal_prob)
    top_class_list.append(top_class)
    top_prob_list.append(top_prob)

metadata['has_animal'] = is_animal_list
metadata['animal_confidence'] = animal_prob_list
metadata['predicted_class'] = top_class_list
metadata['prediction_confidence'] = top_prob_list

In [ ]:
# Filter results
metadata_with_animals = metadata[metadata['has_animal']].copy()
metadata_without_animals = metadata[~metadata['has_animal']].copy()

print(f"\n{'='*60}")
print(f"ANIMAL DETECTION RESULTS")
print(f"{'='*60}")
print(f"Images with animals: {len(metadata_with_animals)} ({len(metadata_with_animals)/len(metadata)*100:.1f}%)")
print(f"Images without animals: {len(metadata_without_animals)} ({len(metadata_without_animals)/len(metadata)*100:.1f}%)")
print(f"Average animal confidence: {metadata_with_animals['animal_confidence'].mean():.3f}")
print(f"Average prediction confidence: {metadata_with_animals['prediction_confidence'].mean():.3f}")
print(f"{'='*60}\n")

In [ ]:
# Visualize results - 50 examples each
n_examples = 50
n_cols = 10
n_rows = 5

fig, axes = plt.subplots(n_rows * 2, n_cols, figsize=(25, 12))

# Top half: Images WITH animals (kept)
metadata_with_animals['animal_confidence'] = metadata_with_animals['animal_confidence'].astype(float)
sample_with = metadata_with_animals.sort_values('animal_confidence', ascending=False).head(n_examples)

for idx, (_, row) in enumerate(sample_with.iterrows()):
    row_idx = idx // n_cols
    col_idx = idx % n_cols
    
    img = PilImage.open(os.path.join(directory, row['file_path']))
    axes[row_idx, col_idx].imshow(img)
    axes[row_idx, col_idx].set_title(
        f"{row['animal_confidence']:.2f}", 
        fontsize=6, color='green', fontweight='bold'
    )
    axes[row_idx, col_idx].axis('off')

# Bottom half: Images WITHOUT animals (removed)
if len(metadata_without_animals) > 0:
    metadata_without_animals['animal_confidence'] = metadata_without_animals['animal_confidence'].astype(float)
    sample_without = metadata_without_animals.sort_values('animal_confidence', ascending=True).head(n_examples)
    
    for idx, (_, row) in enumerate(sample_without.iterrows()):
        row_idx = n_rows + (idx // n_cols)
        col_idx = idx % n_cols
        
        img = PilImage.open(os.path.join(directory, row['file_path']))
        axes[row_idx, col_idx].imshow(img)
        axes[row_idx, col_idx].set_title(
            f"{row['animal_confidence']:.2f}", 
            fontsize=6, color='red', fontweight='bold'
        )
        axes[row_idx, col_idx].axis('off')

# Add section labels
fig.text(0.02, 0.75, 'KEPT (WITH ANIMALS)', fontsize=14, fontweight='bold', 
         color='green', rotation=90, va='center')
fig.text(0.02, 0.25, 'REMOVED (NO ANIMALS)', fontsize=14, fontweight='bold', 
         color='red', rotation=90, va='center')

plt.suptitle('Animal Detection Results - Top 50 Each Category', 
             fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout(rect=[0.03, 0, 1, 0.97])
plt.savefig('animal_detection_grid.png', dpi=200, bbox_inches='tight')
plt.show()

print(f"\nShowing {len(sample_with)} kept images and {len(sample_without)} removed images")

In [ ]:
# Show 20 random non-animal images for manual inspection
n_random = 20
n_cols = 5
n_rows = 4

if len(metadata_without_animals) >= n_random:
    sample_random = metadata_without_animals.sample(n_random)
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 12))
    axes = axes.flatten()
    
    for idx, (_, row) in enumerate(sample_random.iterrows()):
        img = PilImage.open(os.path.join(directory, row['file_path']))
        axes[idx].imshow(img)
        
        # Get just the filename
        filename = os.path.basename(row['file_path'])
        
        axes[idx].set_title(
            f"{filename}\n"
            f"Conf: {row['animal_confidence']:.3f}\n"
            f"{row['predicted_class'][:20]}", 
            fontsize=7, color='red'
        )
        axes[idx].axis('off')
    
    plt.suptitle('Random Sample: 20 Non-Animal Images', fontsize=14, fontweight='bold', color='red')
    plt.tight_layout()
    plt.show()
    
    print(f"Showing 20 random non-animal images out of {len(metadata_without_animals)} total")
    print("Run this cell again to see a different random sample!")
else:
    print(f"Only {len(metadata_without_animals)} non-animal images available")

In [ ]:
# Isolation Forest Outlier Detection
from sklearn.ensemble import IsolationForest

print("="*60)
print("ISOLATION FOREST OUTLIER DETECTION")
print("="*60)

# Prepare embedding data for Isolation Forest
# Convert embeddings to numpy array
embeddings_array = np.stack(good_images_df['embedding'].values)

print(f"Running Isolation Forest on {len(embeddings_array)} images...")
print(f"Embedding dimension: {embeddings_array.shape[1]}")

# Initialize Isolation Forest
# contamination: expected proportion of outliers (e.g., 0.05 = 5%)
iso_forest = IsolationForest(
    contamination=0.05,      # Adjust this based on expected outlier percentage
    random_state=42,
    n_estimators=100,        # Number of trees
    max_samples='auto',
    verbose=0,
    n_jobs=-1                # Use all CPU cores
)

# Fit and predict (-1 for outliers, 1 for inliers)
predictions = iso_forest.fit_predict(embeddings_array)

# Get anomaly scores (lower = more anomalous)
anomaly_scores = iso_forest.score_samples(embeddings_array)

# Add results to dataframe
good_images_df['isolation_forest_prediction'] = predictions
good_images_df['isolation_forest_score'] = anomaly_scores

# Separate outliers and inliers
iso_outliers = good_images_df[good_images_df['isolation_forest_prediction'] == -1].copy()
iso_inliers = good_images_df[good_images_df['isolation_forest_prediction'] == 1].copy()

print(f"\nResults:")
print(f"  Inliers:  {len(iso_inliers)} ({len(iso_inliers)/len(good_images_df)*100:.1f}%)")
print(f"  Outliers: {len(iso_outliers)} ({len(iso_outliers)/len(good_images_df)*100:.1f}%)")
print(f"  Average anomaly score: {anomaly_scores.mean():.4f}")
print(f"  Outlier score range: [{anomaly_scores.min():.4f}, {anomaly_scores.max():.4f}]")
print("="*60)

In [ ]:
# Visualize Isolation Forest outliers
n_examples = 50
n_cols = 10
n_rows = 5

# Sort by anomaly score (most anomalous first)
iso_outliers_sorted = iso_outliers.sort_values('isolation_forest_score', ascending=True)
iso_inliers_sorted = iso_inliers.sort_values('isolation_forest_score', ascending=False)

fig, axes = plt.subplots(n_rows * 2, n_cols, figsize=(25, 12))

# Top half: Most anomalous outliers
sample_outliers = iso_outliers_sorted.head(n_examples)
for idx, (_, row) in enumerate(sample_outliers.iterrows()):
    if idx >= n_examples:
        break
    row_idx = idx // n_cols
    col_idx = idx % n_cols
    
    img = PilImage.open(os.path.join(directory, row['file_path']))
    axes[row_idx, col_idx].imshow(img)
    axes[row_idx, col_idx].set_title(
        f"{row['family'][:10]}\n{row['isolation_forest_score']:.3f}", 
        fontsize=6, color='red', fontweight='bold'
    )
    axes[row_idx, col_idx].axis('off')

# Bottom half: Most typical inliers
sample_inliers = iso_inliers_sorted.head(n_examples)
for idx, (_, row) in enumerate(sample_inliers.iterrows()):
    if idx >= n_examples:
        break
    row_idx = n_rows + (idx // n_cols)
    col_idx = idx % n_cols
    
    img = PilImage.open(os.path.join(directory, row['file_path']))
    axes[row_idx, col_idx].imshow(img)
    axes[row_idx, col_idx].set_title(
        f"{row['family'][:10]}\n{row['isolation_forest_score']:.3f}", 
        fontsize=6, color='green', fontweight='bold'
    )
    axes[row_idx, col_idx].axis('off')

# Add section labels
fig.text(0.02, 0.75, 'OUTLIERS (Anomalous)', fontsize=14, fontweight='bold', 
         color='red', rotation=90, va='center')
fig.text(0.02, 0.25, 'INLIERS (Typical)', fontsize=14, fontweight='bold', 
         color='green', rotation=90, va='center')

plt.suptitle('Isolation Forest Results - Top 50 Each Category', 
             fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout(rect=[0.03, 0, 1, 0.97])
plt.savefig('isolation_forest_results.png', dpi=200, bbox_inches='tight')
plt.show()

print(f"\nShowing {len(sample_outliers)} outliers and {len(sample_inliers)} inliers")

In [ ]:
# Local Outlier Factor (LOF) Outlier Detection
from sklearn.neighbors import LocalOutlierFactor

print("="*60)
print("LOCAL OUTLIER FACTOR (LOF) OUTLIER DETECTION")
print("="*60)

# Prepare embedding data
embeddings_array = np.stack(good_images_df['embedding'].values)

print(f"Running LOF on {len(embeddings_array)} images...")
print(f"Embedding dimension: {embeddings_array.shape[1]}")

# Initialize LOF
# contamination: expected proportion of outliers
# n_neighbors: number of neighbors to consider (smaller = more local, larger = more global)
lof = LocalOutlierFactor(
    n_neighbors=20,           # Adjust based on dataset size
    contamination=0.05,       # 5% outliers
    novelty=False,            # Set to False for outlier detection on training data
    n_jobs=-1
)

# Fit and predict (-1 for outliers, 1 for inliers)
predictions = lof.fit_predict(embeddings_array)

# Get negative outlier factor scores (more negative = more anomalous)
lof_scores = lof.negative_outlier_factor_

# Add results to dataframe
good_images_df['lof_prediction'] = predictions
good_images_df['lof_score'] = lof_scores

# Separate outliers and inliers
lof_outliers = good_images_df[good_images_df['lof_prediction'] == -1].copy()
lof_inliers = good_images_df[good_images_df['lof_prediction'] == 1].copy()

print(f"\nResults:")
print(f"  Inliers:  {len(lof_inliers)} ({len(lof_inliers)/len(good_images_df)*100:.1f}%)")
print(f"  Outliers: {len(lof_outliers)} ({len(lof_outliers)/len(good_images_df)*100:.1f}%)")
print(f"  Average LOF score: {lof_scores.mean():.4f}")
print(f"  Outlier score range: [{lof_scores.min():.4f}, {lof_scores.max():.4f}]")
print("="*60)

# Visualize results
n_examples = 50
n_cols = 10
n_rows = 5

lof_outliers_sorted = lof_outliers.sort_values('lof_score', ascending=True)
lof_inliers_sorted = lof_inliers.sort_values('lof_score', ascending=False)

fig, axes = plt.subplots(n_rows * 2, n_cols, figsize=(25, 12))

# Top half: Most anomalous outliers
sample_outliers = lof_outliers_sorted.head(n_examples)
for idx, (_, row) in enumerate(sample_outliers.iterrows()):
    if idx >= n_examples:
        break
    row_idx = idx // n_cols
    col_idx = idx % n_cols
    
    img = PilImage.open(os.path.join(directory, row['file_path']))
    axes[row_idx, col_idx].imshow(img)
    axes[row_idx, col_idx].set_title(
        f"{row['family'][:10]}\n{row['lof_score']:.3f}", 
        fontsize=6, color='red', fontweight='bold'
    )
    axes[row_idx, col_idx].axis('off')

# Bottom half: Most typical inliers
sample_inliers = lof_inliers_sorted.head(n_examples)
for idx, (_, row) in enumerate(sample_inliers.iterrows()):
    if idx >= n_examples:
        break
    row_idx = n_rows + (idx // n_cols)
    col_idx = idx % n_cols
    
    img = PilImage.open(os.path.join(directory, row['file_path']))
    axes[row_idx, col_idx].imshow(img)
    axes[row_idx, col_idx].set_title(
        f"{row['family'][:10]}\n{row['lof_score']:.3f}", 
        fontsize=6, color='green', fontweight='bold'
    )
    axes[row_idx, col_idx].axis('off')

fig.text(0.02, 0.75, 'OUTLIERS (Anomalous)', fontsize=14, fontweight='bold', 
         color='red', rotation=90, va='center')
fig.text(0.02, 0.25, 'INLIERS (Typical)', fontsize=14, fontweight='bold', 
         color='green', rotation=90, va='center')

plt.suptitle('Local Outlier Factor Results - Top 50 Each Category', 
             fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout(rect=[0.03, 0, 1, 0.97])
plt.show()

In [ ]:
# DBSCAN Outlier Detection
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler

print("="*60)
print("DBSCAN OUTLIER DETECTION")
print("="*60)

# Prepare and normalize embeddings
embeddings_array = np.stack(good_images_df['embedding'].values)

# Normalize embeddings for better distance metrics
scaler = StandardScaler()
embeddings_normalized = scaler.fit_transform(embeddings_array)

print(f"Running DBSCAN on {len(embeddings_array)} images...")

# Initialize DBSCAN
# eps: maximum distance between samples to be considered neighbors
# min_samples: minimum samples in a neighborhood to form a cluster
dbscan = DBSCAN(
    eps=3.0,              # Adjust based on your data (try values like 2.5, 3.0, 3.5)
    min_samples=5,        # Minimum points to form a cluster
    metric='euclidean',
    n_jobs=-1
)

# Fit and get cluster labels (-1 means outlier/noise)
cluster_labels = dbscan.fit_predict(embeddings_normalized)

# Add results to dataframe
good_images_df['dbscan_cluster'] = cluster_labels
good_images_df['dbscan_is_outlier'] = (cluster_labels == -1)

# Separate outliers and inliers
dbscan_outliers = good_images_df[good_images_df['dbscan_is_outlier'] == True].copy()
dbscan_inliers = good_images_df[good_images_df['dbscan_is_outlier'] == False].copy()

n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
n_noise = list(cluster_labels).count(-1)

print(f"\nResults:")
print(f"  Number of clusters: {n_clusters}")
print(f"  Inliers:  {len(dbscan_inliers)} ({len(dbscan_inliers)/len(good_images_df)*100:.1f}%)")
print(f"  Outliers: {len(dbscan_outliers)} ({len(dbscan_outliers)/len(good_images_df)*100:.1f}%)")
print(f"  Noise points: {n_noise}")
print("="*60)

# Visualize outliers
n_examples = min(50, len(dbscan_outliers))
n_cols = 10
n_rows = (n_examples + n_cols - 1) // n_cols

if len(dbscan_outliers) > 0:
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(25, 3 * n_rows))
    axes = axes.flatten() if n_rows > 1 else [axes] if n_cols == 1 else axes
    
    sample_outliers = dbscan_outliers.sample(min(n_examples, len(dbscan_outliers)))
    
    for idx, (_, row) in enumerate(sample_outliers.iterrows()):
        img = PilImage.open(os.path.join(directory, row['file_path']))
        axes[idx].imshow(img)
        axes[idx].set_title(f"{row['family'][:12]}", fontsize=7, color='red')
        axes[idx].axis('off')
    
    # Hide unused subplots
    for idx in range(len(sample_outliers), len(axes)):
        axes[idx].axis('off')
    
    plt.suptitle('DBSCAN Outliers (Noise Points)', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("No outliers detected by DBSCAN")

### Outlier removal

### Import the dataset

In [ ]:
# Stratified split the metadata to get the paths for the images
train_df, test_df = train_test_split(
  good_images_df,   # use only good images
  test_size=0.1,
  stratify=good_images_df['family'],
  shuffle=True,
  random_state=42
)

train_df, val_df = train_test_split(
  train_df,
  test_size=0.1,
  stratify=train_df['family'],
  shuffle=True,
  random_state=42
)

# Print the final proportions of the split
print(f"Training  : {len(train_df)} ({round(len(train_df)/len(metadata)*100)}%) \
        \nValidation: {len(val_df)} ({round(len(val_df)/len(metadata)*100)}%)  \
        \nTesting   : {len(test_df)} ({round(len(test_df)/len(metadata)*100)}%) \
        \nTotal     : {len(train_df) + len(val_df) + len(test_df)}")

In [ ]:
# Create ImageDataGenerator with ENHANCED augmentation
train_datagen = ImageDataGenerator(
    preprocessing_function=advanced_preprocessing,  # Using enhanced preprocessing!
    rotation_range=30,              # Increased from 25
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    zoom_range=0.25,                # Increased from 0.2
    shear_range=0.15,
    fill_mode='nearest'
)

# Validation/Test generator (no augmentation, only preprocessing)
val_test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    directory=directory,
    x_col='file_path',
    y_col='family',
    target_size=image_size,
    batch_size=32,
    class_mode='sparse',
    color_mode='rgb',
    shuffle=True
)

val_generator = val_test_datagen.flow_from_dataframe(
    dataframe=val_df,
    directory=directory,
    x_col='file_path',
    y_col='family',
    target_size=image_size,
    batch_size=32,
    class_mode='sparse',
    color_mode='rgb',
    shuffle=False
)

test_generator = val_test_datagen.flow_from_dataframe(
    dataframe=test_df,
    directory=directory,
    x_col='file_path',
    y_col='family',
    target_size=image_size,
    batch_size=32,
    class_mode='sparse',
    color_mode='rgb',
    shuffle=False
)

print("✅ Data generators created with enhanced color augmentation")

In [ ]:
# Create ImageDataGenerator for both train and test
# Transform all the images to the image_size and rgb mode
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=25,              # Random rotations
    width_shift_range=0.2,          # Horizontal shifts
    height_shift_range=0.2,         # Vertical shifts
    horizontal_flip=True,           # Mirror images
    zoom_range=0.2,                 # Random zoom
    shear_range=0.15,               # Shear transformations
    brightness_range=[0.8, 1.2],    # Brightness variations
    fill_mode='nearest'             # Fill pixels after transformations
)  #rescale=1./255)

train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    directory=directory,
    x_col='file_path',
    y_col='family',
    target_size=image_size,
    batch_size=32,
    class_mode='sparse',
    color_mode='rgb',
    shuffle=True
)


val_generator = val_test_datagen.flow_from_dataframe(
    dataframe=val_df,
    directory=directory,
    x_col='file_path',
    y_col='family',
    target_size=image_size,
    batch_size=32,
    class_mode='sparse',
    color_mode='rgb',
    shuffle=False
)


test_generator = val_test_datagen.flow_from_dataframe(
    dataframe=test_df,
    directory=directory,
    x_col='file_path',
    y_col='family',
    target_size=image_size,
    batch_size=32,
    class_mode='sparse',
    color_mode='rgb',
    shuffle=False
)

In [ ]:
# Get a batch of images and labels
images, labels = next(train_generator)

# Get class names
class_names = list(train_generator.class_indices.keys())

# Plot the images
plt.figure(figsize=(12, 8))

for i in range(21):
    ax = plt.subplot(3, 7, i + 1)
    plt.imshow(images[i])
    plt.title(class_names[np.argmax(labels[i])])  # Get actual labele names
    plt.axis("off")

plt.tight_layout()
plt.show()

## Model


In [ ]:
N_CLASSES = len(train_generator.class_indices)
print(f"Number of classes: {N_CLASSES}")

### Small custom model

In [ ]:
# Get number of classes
num_classes = len(train_generator.class_indices)
print(f"Number of classes: {num_classes}")

### Transfer learning with keras base model

#### EfficientNetB4

In [ ]:
from tensorflow.keras.applications import EfficientNetB4

In [ ]:
from focal_loss import SparseCategoricalFocalLoss

print("✅ Focal loss imported - will be used for training")

In [ ]:
# from focal_loss import SparseCategoricalFocalLoss

In [ ]:
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_generator.classes),
    y=train_generator.classes
)
class_weight_dict = dict(enumerate(class_weights))

In [ ]:
# Build the classifier with DEEPER HEAD and skip connections
from tensorflow.keras.layers import Add, Activation

x = base_model.output
x = GlobalAveragePooling2D()(x)

# Dense block 1: 1024 neurons
x1 = Dense(1024, kernel_regularizer=L2(0.01))(x)
x1 = BatchNormalization()(x1)
x1 = Activation('relu')(x1)
x1 = Dropout(0.4)(x1)

# Dense block 2: 512 neurons
x2 = Dense(512, kernel_regularizer=L2(0.01))(x1)
x2 = BatchNormalization()(x2)
x2 = Activation('relu')(x2)
x2 = Dropout(0.3)(x2)

# Skip connection (residual connection from pooled features)
x_skip = Dense(512)(x)
x2_combined = Add()([x2, x_skip])

# Output layer (dtype='float32' for mixed precision stability)
output = Dense(N_CLASSES, activation='softmax', dtype='float32')(x2_combined)

# Create model
model = Model(inputs=base_model.input, outputs=output)

print("✅ Model built with deep classification head (3 layers + skip connection)")

In [ ]:
# Compile model with FOCAL LOSS for class imbalance
from tensorflow.keras.callbacks import ModelCheckpoint, TensorBoard

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss=SparseCategoricalFocalLoss(gamma=2.0),  # Focal loss focuses on hard examples!
    metrics=['accuracy'],
)

# Enhanced callbacks with F1 tracking, TensorBoard, and model checkpointing
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=10,  # Increased patience
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,  # More aggressive reduction
        patience=5,
        min_lr=1e-7,
        verbose=1
    ),
    ModelCheckpoint(
        'best_model_stage1.h5',
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    ),
    F1Callback(validation_data=val_generator),
    TensorBoard(
        log_dir=f'logs/{datetime.datetime.now().strftime("%Y%m%d-%H%M%S")}',
        histogram_freq=1
    )
]

print("="*70)
print("STAGE 1: Training classification head only (base model frozen)")
print("="*70)

# Fit the model
history1 = model.fit(
    train_generator,
    epochs=10,  # Stage 1: head only
    class_weight=class_weight_dict,
    validation_data=val_generator,
    callbacks=callbacks,
    verbose=1
)

print(f"\n✅ Stage 1 complete! Best val_loss: {min(history1.history['val_loss']):.4f}")

In [ ]:
# PROGRESSIVE UNFREEZING - Stages 2, 3, 4
# This gradually fine-tunes the pretrained model from top to bottom

print("="*70)
print("STAGE 2: Unfreezing top 50 layers")
print("="*70)

# Unfreeze top 50 layers
base_model.trainable = True
for layer in base_model.layers[:-50]:
    layer.trainable = False

trainable_params = sum([tf.size(w).numpy() for w in model.trainable_weights])
print(f"Trainable parameters: {trainable_params:,}")

model.compile(
    optimizer=Adam(learning_rate=1e-4),  # 10x lower learning rate
    loss=SparseCategoricalFocalLoss(gamma=2.0),
    metrics=['accuracy']
)

history2 = model.fit(
    train_generator,
    epochs=10,
    class_weight=class_weight_dict,
    validation_data=val_generator,
    callbacks=callbacks,
    verbose=1
)

print(f"✅ Stage 2 complete! Best val_loss: {min(history2.history['val_loss']):.4f}")

# ==================== STAGE 3 ====================
print("\n" + "="*70)
print("STAGE 3: Unfreezing top 100 layers")
print("="*70)

# Unfreeze top 100 layers
for layer in base_model.layers[:-100]:
    layer.trainable = False
for layer in base_model.layers[-100:]:
    layer.trainable = True

trainable_params = sum([tf.size(w).numpy() for w in model.trainable_weights])
print(f"Trainable parameters: {trainable_params:,}")

model.compile(
    optimizer=Adam(learning_rate=5e-5),  # Even lower learning rate
    loss=SparseCategoricalFocalLoss(gamma=2.0),
    metrics=['accuracy']
)

history3 = model.fit(
    train_generator,
    epochs=10,
    class_weight=class_weight_dict,
    validation_data=val_generator,
    callbacks=callbacks,
    verbose=1
)

print(f"✅ Stage 3 complete! Best val_loss: {min(history3.history['val_loss']):.4f}")

# ==================== STAGE 4 ====================
print("\n" + "="*70)
print("STAGE 4: Full fine-tuning (all layers unfrozen)")
print("="*70)

# Unfreeze all layers
base_model.trainable = True

trainable_params = sum([tf.size(w).numpy() for w in model.trainable_weights])
print(f"Trainable parameters: {trainable_params:,}")

model.compile(
    optimizer=Adam(learning_rate=1e-5),  # Very low learning rate for full fine-tuning
    loss=SparseCategoricalFocalLoss(gamma=2.0),
    metrics=['accuracy']
)

history4 = model.fit(
    train_generator,
    epochs=10,
    class_weight=class_weight_dict,
    validation_data=val_generator,
    callbacks=callbacks,
    verbose=1
)

print(f"✅ Stage 4 complete! Best val_loss: {min(history4.history['val_loss']):.4f}")

# Save final model
model.save('final_model_progressive_unfreezing.h5')
print("\n🎉 Progressive unfreezing complete! Model saved as 'final_model_progressive_unfreezing.h5'")

In [ ]:
# Compile with categorical crossentropy for multi-class
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

# Set up callbacks
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.3,
        patience=3,
        min_lr=1e-7,
        verbose=1
    )
]

# Fit the model (batch_size removed - using generator's batch_size of 32)
history1 = model.fit(
    train_generator,
    epochs=40,
    class_weight=class_weight_dict,
    validation_data=val_generator,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# # Fine-tuning the model
# # Unfreeze base model
# base_model.trainable = True

# # Recompile with much lower learning rate
# model.compile(
#     optimizer=Adam(learning_rate=1e-5),  # 100x lower learning rate
#     loss='categorical_crossentropy',
#     metrics=['accuracy', 'top_k_categorical_accuracy']
# )

# history2 = model.fit(
#     train_generator,
#     epochs=20,
#     validation_data=val_generator,
#     callbacks=callbacks,
#     verbose=1
# )

#### EfficientNetB0

In [ ]:
from tensorflow.keras.applications import EfficientNetB0

In [ ]:
base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(*image_size, 3) 
)

# Freeze base model initially
base_model.trainable = False

In [ ]:
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_generator.classes),
    y=train_generator.classes
)
class_weight_dict = dict(enumerate(class_weights))

In [ ]:
# Build the classifier
with tf.device('/GPU:0'):   # Move the model to gpu
    model = Sequential([
        base_model,
        GlobalAveragePooling2D(),
        Dropout(0.4),
        BatchNormalization(),
        Dense(N_CLASSES, kernel_regularizer=L2(0.02), activation='softmax')
    ])

In [ ]:
# See the parameter counts
print("Total params:", model.count_params())
print("Trainable params:", sum([tf.size(w).numpy() for w in model.trainable_weights]))
print("Non-trainable params:", sum([tf.size(w).numpy() for w in model.non_trainable_weights]))

## Results

In [ ]:
# COMPREHENSIVE EVALUATION FUNCTION
def comprehensive_evaluation(model, test_generator):
    """
    Generate detailed evaluation report with insights for the project
    """
    
    # Get predictions
    print("="*70)
    print("GENERATING PREDICTIONS...")
    print("="*70)
    y_pred_probs = model.predict(test_generator, verbose=1)
    y_pred = np.argmax(y_pred_probs, axis=1)
    y_true = test_generator.classes
    class_names = list(test_generator.class_indices.keys())
    
    # ==================== OVERALL METRICS ====================
    print("\n" + "="*70)
    print("OVERALL METRICS")
    print("="*70)
    accuracy = (y_pred == y_true).mean()
    f1_macro = f1_score(y_true, y_pred, average='macro')
    f1_weighted = f1_score(y_true, y_pred, average='weighted')
    f1_micro = f1_score(y_true, y_pred, average='micro')
    
    print(f"Accuracy:     {accuracy:.4f}")
    print(f"F1 Macro:     {f1_macro:.4f}")
    print(f"F1 Weighted:  {f1_weighted:.4f}")
    print(f"F1 Micro:     {f1_micro:.4f}")
    
    # ==================== PER-CLASS REPORT ====================
    print("\n" + "="*70)
    print("PER-CLASS CLASSIFICATION REPORT")
    print("="*70)
    report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
    report_df = pd.DataFrame(report).transpose()
    
    # Best performing classes
    print("\n📊 TOP 10 BEST PERFORMING CLASSES:")
    best_classes = report_df.iloc[:-3].sort_values('f1-score', ascending=False).head(10)
    print(best_classes[['precision', 'recall', 'f1-score', 'support']])
    
    # Worst performing classes
    print("\n❌ TOP 10 WORST PERFORMING CLASSES:")
    worst_classes = report_df.iloc[:-3].sort_values('f1-score', ascending=True).head(10)
    print(worst_classes[['precision', 'recall', 'f1-score', 'support']])
    
    # ==================== CONFIDENCE ANALYSIS ====================
    print("\n" + "="*70)
    print("CONFIDENCE ANALYSIS")
    print("="*70)
    max_probs = np.max(y_pred_probs, axis=1)
    correct_mask = (y_pred == y_true)
    
    print(f"Avg confidence (correct):   {max_probs[correct_mask].mean():.4f}")
    print(f"Avg confidence (incorrect): {max_probs[~correct_mask].mean():.4f}")
    print(f"Confidence gap: {max_probs[correct_mask].mean() - max_probs[~correct_mask].mean():.4f}")
    
    # Visualize confidence distributions
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: Confidence distribution
    axes[0].hist(max_probs[correct_mask], bins=50, alpha=0.7, label='Correct', color='green', edgecolor='black')
    axes[0].hist(max_probs[~correct_mask], bins=50, alpha=0.7, label='Incorrect', color='red', edgecolor='black')
    axes[0].set_xlabel('Confidence', fontsize=12)
    axes[0].set_ylabel('Count', fontsize=12)
    axes[0].set_title('Confidence Distribution', fontsize=14, fontweight='bold')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Plot 2: Accuracy vs confidence threshold
    thresholds = np.linspace(0, 1, 20)
    accs = []
    coverage = []
    for t in thresholds:
        mask = max_probs >= t
        coverage.append(mask.sum() / len(mask) * 100)  # Percentage
        if mask.sum() > 0:
            accs.append((y_pred[mask] == y_true[mask]).mean() * 100)  # Percentage
        else:
            accs.append(0)
    
    ax2 = axes[1]
    line1 = ax2.plot(thresholds, accs, 'b-', linewidth=2, label='Accuracy', marker='o')
    ax2.set_xlabel('Confidence Threshold', fontsize=12)
    ax2.set_ylabel('Accuracy (%)', color='b', fontsize=12)
    ax2.tick_params(axis='y', labelcolor='b')
    ax2.grid(True, alpha=0.3)
    
    ax2_twin = ax2.twinx()
    line2 = ax2_twin.plot(thresholds, coverage, 'r--', linewidth=2, label='Coverage', marker='s')
    ax2_twin.set_ylabel('Coverage (%)', color='r', fontsize=12)
    ax2_twin.tick_params(axis='y', labelcolor='r')
    
    axes[1].set_title('Accuracy vs Confidence Threshold', fontsize=14, fontweight='bold')
    
    # Add combined legend
    lines = line1 + line2
    labels = [l.get_label() for l in lines]
    ax2.legend(lines, labels, loc='center right')
    
    plt.tight_layout()
    plt.savefig('confidence_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # ==================== TOP CONFUSIONS ====================
    print("\n" + "="*70)
    print("TOP 10 MOST COMMON MISCLASSIFICATIONS")
    print("="*70)
    cm = confusion_matrix(y_true, y_pred)
    cm_no_diag = cm.copy()
    np.fill_diagonal(cm_no_diag, 0)
    
    top_confusions = []
    for i in range(10):
        max_idx = np.unravel_index(np.argmax(cm_no_diag), cm_no_diag.shape)
        count = cm_no_diag[max_idx]
        if count == 0:
            break
        true_class = class_names[max_idx[0]]
        pred_class = class_names[max_idx[1]]
        print(f"{i+1:2d}. {true_class:25s} → {pred_class:25s}: {count:3d} times")
        top_confusions.append((true_class, pred_class, count))
        cm_no_diag[max_idx] = 0
    
    print("\n" + "="*70)
    print("✅ COMPREHENSIVE EVALUATION COMPLETE")
    print("="*70)
    
    return report_df, top_confusions, y_pred, y_pred_probs

# Run comprehensive evaluation
class_names = list(test_generator.class_indices.keys())
results_df, confusions, y_pred_classes, y_pred_probs = comprehensive_evaluation(model, test_generator)

In [ ]:
# Generate Grad-CAM visualizations for correct and incorrect predictions
print("="*70)
print("GRAD-CAM VISUALIZATIONS")
print("="*70)

# Get y_true for comparison
y_true = test_generator.classes

# Get indices of correct and incorrect predictions
correct_indices = np.where(y_pred_classes == y_true)[0]
incorrect_indices = np.where(y_pred_classes != y_true)[0]

# Show examples of CORRECT predictions
print("\n✅ EXAMPLES OF CORRECT PREDICTIONS WITH GRAD-CAM:")
print("-" * 70)
for i in range(min(3, len(correct_indices))):
    idx = correct_indices[i]
    img_path = directory + test_df.iloc[idx]['file_path']
    true_label = class_names[y_true[idx]]
    print(f"\nExample {i+1}: {true_label}")
    display_gradcam(model, img_path, class_names, true_label)

# Show examples of INCORRECT predictions
print("\n❌ EXAMPLES OF INCORRECT PREDICTIONS WITH GRAD-CAM:")
print("-" * 70)
for i in range(min(3, len(incorrect_indices))):
    idx = incorrect_indices[i]
    img_path = directory + test_df.iloc[idx]['file_path']
    true_label = class_names[y_true[idx]]
    print(f"\nExample {i+1}: True = {true_label}")
    display_gradcam(model, img_path, class_names, true_label)

print("\n" + "="*70)
print("✅ GRAD-CAM VISUALIZATION COMPLETE")
print("="*70)

In [ ]:
# GRAD-CAM VISUALIZATION - Show where the model is looking
def make_gradcam(model, img_array, pred_index=None):
    """
    Generate Grad-CAM heatmap showing which parts of the image the model focuses on
    """
    # Find last convolutional layer
    last_conv_layer = None
    for layer in reversed(model.layers):
        if 'conv' in layer.name.lower() and len(layer.output_shape) == 4:
            last_conv_layer = layer.name
            break
    
    if last_conv_layer is None:
        # For EfficientNet, try to find by checking the base model
        for layer in reversed(model.layers):
            if hasattr(layer, 'layers'):  # It's a base model
                for sub_layer in reversed(layer.layers):
                    if 'conv' in sub_layer.name.lower() and len(sub_layer.output_shape) == 4:
                        last_conv_layer = sub_layer.name
                        break
                if last_conv_layer:
                    break
    
    if last_conv_layer is None:
        print("⚠️ No convolutional layer found")
        return None
    
    print(f"Using layer: {last_conv_layer}")
    
    # Create gradient model
    try:
        grad_model = tf.keras.models.Model(
            [model.inputs],
            [model.get_layer(last_conv_layer).output, model.output]
        )
    except:
        # If that fails, try accessing through base_model
        base_layer = model.get_layer('efficientnetb4')
        grad_model = tf.keras.models.Model(
            [model.inputs],
            [base_layer.get_layer(last_conv_layer).output, model.output]
        )
    
    # Compute gradients
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]
    
    # Get gradients
    grads = tape.gradient(class_channel, conv_outputs)
    
    # Global average pooling
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    
    # Multiply
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    
    # Normalize
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-10)
    
    return heatmap.numpy()

def display_gradcam(model, img_path, class_names, true_label=None):
    """Display image with Grad-CAM overlay"""
    # Load and preprocess
    img = tf.keras.preprocessing.image.load_img(img_path, target_size=(500, 375))
    img_array = tf.keras.preprocessing.image.img_to_array(img)
    img_array = preprocess_input(img_array)
    img_array_batch = np.expand_dims(img_array, axis=0)
    
    # Predict
    pred = model.predict(img_array_batch, verbose=0)
    pred_class_idx = np.argmax(pred[0])
    pred_prob = pred[0][pred_class_idx]
    pred_class_name = class_names[pred_class_idx]
    
    # Generate Grad-CAM
    heatmap = make_gradcam(model, img_array_batch, pred_class_idx)
    
    if heatmap is None:
        print("Could not generate Grad-CAM")
        return
    
    # Resize heatmap to image size
    heatmap = cv2.resize(heatmap, (375, 500))
    heatmap = np.uint8(255 * heatmap)
    heatmap_colored = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    
    # Overlay
    img_bgr = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
    superimposed = cv2.addWeighted(img_bgr, 0.6, heatmap_colored, 0.4, 0)
    
    # Display
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    axes[0].imshow(img)
    title = 'Original Image'
    if true_label:
        title += f'\n✅ True: {true_label}'
    axes[0].set_title(title, fontsize=12, fontweight='bold')
    axes[0].axis('off')
    
    axes[1].imshow(heatmap, cmap='jet')
    axes[1].set_title('Grad-CAM\n(Model Attention)', fontsize=12, fontweight='bold')
    axes[1].axis('off')
    
    axes[2].imshow(cv2.cvtColor(superimposed, cv2.COLOR_BGR2RGB))
    title = f'Overlay\n🔮 Pred: {pred_class_name}\nConf: {pred_prob:.3f}'
    axes[2].set_title(title, fontsize=12, fontweight='bold')
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()

print("✅ Grad-CAM functions defined")

In [ ]:
plt.plot(history1.history['accuracy'])
plt.plot(history1.history['val_accuracy'])
plt.title('Model Acc')
plt.ylabel('Loss')
plt.xlabel('Epochs')
plt.legend(['train', 'validation'])
plt.show()

In [ ]:
test_results = model.evaluate(test_generator, verbose=1)
print(f"Test Accuracy: {test_results[1]:.4f}")
print(f"Test Loss: {test_results[0]:.4f}")

In [ ]:
# Get predictions for F1 score
y_pred = model.predict(test_generator, verbose=1)
y_pred_classes = np.argmax(y_pred, axis=1)

# Get true labels
y_true = test_generator.classes

# Calculate F1 scores
f1_micro = f1_score(y_true, y_pred_classes, average='micro')
f1_macro = f1_score(y_true, y_pred_classes, average='macro')
f1_weighted = f1_score(y_true, y_pred_classes, average='weighted')

print(f"\nF1 Score (Micro): {f1_micro:.4f}")
print(f"F1 Score (Macro): {f1_macro:.4f}")
print(f"F1 Score (Weighted): {f1_weighted:.4f}")

#Detailed classification report
# print(classification_report(y_true, y_pred_classes, target_names=test_generator.class_indices.keys()))

In [ ]:
# ============================================================================
# FINAL PROJECT SUMMARY
# ============================================================================

print("="*80)
print(" " * 25 + "FINAL PROJECT SUMMARY")
print("="*80)

print("\n📊 DATASET INFORMATION:")
print("-" * 80)
print(f"  Total images (original):          {len(metadata):,}")
print(f"  After outlier removal:            {len(good_images_df):,} ({len(good_images_df)/len(metadata)*100:.1f}%)")
print(f"  Number of classes (families):     {N_CLASSES}")
print(f"  Number of phyla:                  {metadata['phylum'].nunique()}")
print(f"  Train/Val/Test split:             {len(train_df):,} / {len(val_df):,} / {len(test_df):,}")
print(f"  Image size (standardized):        {image_size}")

print("\n🔍 DATA PREPROCESSING:")
print("-" * 80)
print("  Outlier Detection Methods:")
print("    ✓ Distance to centroid (iterative, 2 rounds)")
print("    ✓ ImageNet validation (InceptionV3)")
print("    ✓ Isolation Forest (contamination=0.05)")
print("    ✓ Local Outlier Factor (LOF, n_neighbors=20)")
print("    ✓ DBSCAN clustering (eps=3.0, min_samples=5)")
print(f"  Total outliers removed:           {len(metadata) - len(good_images_df):,} ({(len(metadata) - len(good_images_df))/len(metadata)*100:.1f}%)")
print("\n  Image Augmentation:")
print("    ✓ Rotation (±30°)")
print("    ✓ Width/Height shift (±20%)")
print("    ✓ Horizontal flip")
print("    ✓ Zoom (±25%)")
print("    ✓ Shear (15%)")
print("    ✓ Random brightness (±20%)")
print("    ✓ Random contrast (0.8-1.2)")
print("    ✓ Random saturation (0.8-1.2)")

print("\n🏗️ MODEL ARCHITECTURE:")
print("-" * 80)
print("  Backbone:                         EfficientNetB4 (pretrained on ImageNet)")
print("  Classification Head:")
print("    ├─ GlobalAveragePooling2D")
print("    ├─ Dense(1024) + BatchNorm + ReLU + Dropout(0.4)")
print("    ├─ Dense(512) + BatchNorm + ReLU + Dropout(0.3)")
print("    ├─ Skip connection from pooled features")
print("    └─ Dense(202, softmax, dtype=float32)")
print(f"  Total parameters:                 {model.count_params():,}")
print(f"  Trainable parameters (final):     {sum([tf.size(w).numpy() for w in model.trainable_weights]):,}")

print("\n📈 TRAINING STRATEGY:")
print("-" * 80)
print("  Loss Function:                    Sparse Categorical Focal Loss (gamma=2.0)")
print("  Optimizer:                        Adam")
print("  Class Weighting:                  Balanced (computed from training set)")
print("  Mixed Precision:                  Enabled (float16)")
print("  XLA Compilation:                  Enabled")
print("\n  Progressive Unfreezing (4 stages):")
print("    Stage 1: Head only           | LR: 1e-3  | Epochs: 10")
print("    Stage 2: Top 50 layers       | LR: 1e-4  | Epochs: 10")
print("    Stage 3: Top 100 layers      | LR: 5e-5  | Epochs: 10")
print("    Stage 4: All layers          | LR: 1e-5  | Epochs: 10")
print(f"  Total training epochs:            {len(history1.history['loss']) + len(history2.history['loss']) + len(history3.history['loss']) + len(history4.history['loss'])}")

print("\n📊 CALLBACKS:")
print("-" * 80)
print("  ✓ EarlyStopping (patience=10, restore_best_weights)")
print("  ✓ ReduceLROnPlateau (factor=0.5, patience=5)")
print("  ✓ ModelCheckpoint (save best model)")
print("  ✓ F1 Score tracking (weighted & macro)")
print("  ✓ TensorBoard logging (with histograms)")

print("\n✅ FINAL RESULTS:")
print("-" * 80)
# Get final metrics from comprehensive evaluation
final_acc = results_df.loc['accuracy', 'precision']  # Accuracy is in the precision column for 'accuracy' row
final_f1_weighted = results_df.loc['weighted avg', 'f1-score']
final_f1_macro = results_df.loc['macro avg', 'f1-score']

print(f"  Test Accuracy:                    {final_acc:.4f} ({final_acc*100:.2f}%)")
print(f"  F1 Score (Weighted):              {final_f1_weighted:.4f}")
print(f"  F1 Score (Macro):                 {final_f1_macro:.4f}")
print(f"  F1 Score (Micro):                 {results_df.loc['accuracy', 'recall']:.4f}")  # Micro F1 = Accuracy

print("\n🔬 ANALYSIS COMPLETED:")
print("-" * 80)
print("  ✓ Comprehensive per-class metrics (all 202 classes)")
print("  ✓ Confusion matrix analysis")
print("  ✓ Confidence calibration analysis")
print("  ✓ Top 10 best/worst performing classes")
print("  ✓ Top 10 most common misclassifications")
print("  ✓ Grad-CAM visualizations (correct & incorrect predictions)")

print("\n💡 INNOVATIONS IMPLEMENTED:")
print("-" * 80)
print("  ✓ Multi-method outlier detection pipeline (5 complementary techniques)")
print("  ✓ Deep classification head with skip connections")
print("  ✓ Progressive unfreezing strategy (4-stage)")
print("  ✓ Focal loss for class imbalance")
print("  ✓ Enhanced data augmentation (geometric + color)")
print("  ✓ F1 score tracking during training")
print("  ✓ Grad-CAM interpretability")
print("  ✓ Confidence calibration analysis")

print("\n📁 FILES GENERATED:")
print("-" * 80)
print("  ✓ best_model_stage1.h5            (Best model from stage 1)")
print("  ✓ final_model_progressive_unfreezing.h5  (Final trained model)")
print("  ✓ confidence_analysis.png         (Confidence distribution plots)")
print("  ✓ logs/                           (TensorBoard training logs)")

print("\n" + "="*80)
print(" " * 22 + "🎉 PROJECT COMPLETE! 🎉")
print("="*80)
print("\nModel is ready for evaluation in the report!")
print("Expected improvements over baseline:")
print(f"  • Baseline (linear classifier):   ~79% accuracy")
print(f"  • Current model:                  ~{final_acc*100:.0f}% accuracy (expected: 85-90%)")
print(f"  • Improvement:                    +{(final_acc - 0.79)*100:.0f} percentage points")
print("="*80)

In [ ]:
# Create confusion matrix
cm = confusion_matrix(y_true, y_pred_classes)

# Plot confusion matrix
plt.figure(figsize=(20, 16))
sns.heatmap(cm, annot=False, fmt='d', cmap='Blues', cbar=True)         # annot=np.where(cm > 0, cm, ""), annot_kws={"size": 5}
plt.title('Confusion Matrix: Predictions vs Actual Labels')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

# Print shape and summary
print(f"Confusion Matrix Shape: {cm.shape}")
print(f"Total Predictions: {cm.sum()}")

In [ ]:
# y_pred_classes[1]
dict(test_generator.class_indices.items())[low_images_class]

In [ ]:
def class_metrics(class_name):

    class_idx = test_generator.class_indices[class_name]

    TP = int(cm[class_idx, class_idx])
    FN = int(cm[class_idx, :].sum() - TP)   # actual positives missed
    FP = int(cm[:, class_idx].sum() - TP)   # predicted as this class but not actual
    TN = int(cm.sum() - (TP + FP + FN))
    support = TP + FN
    
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0   #per-class accuracy / sensitivity
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    class_accuracy = (TP + TN) / cm.sum()               # accuracy treating this class vs rest
    
    # class_name =idx_to_class[class_idx]
    print(f"Class: {class_name} (index={class_idx})")
    print(f" Support (true samples) : {support}")
    print(f" TP: {TP} | FN: {FN} | FP: {FP} | TN: {TN}")
    print(f" Precision : {precision:.4f}")
    print(f" Recall    : {recall:.4f}")     
    print(f" F1-score  : {f1:.4f}")
    print(f" Class-wise accuracy : {class_accuracy:.4f}")



In [ ]:
class_metrics(low_images_class)